# Interpretable Member Attrition Modeling

This historical portfolio notebook develops an interpretable logistic-regression model for member attrition using anonymized account and transaction features.

The original database connection and source query have been removed. The notebook expects a prepared local extract named `member_attrition_features.csv`. Original data is not distributed.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
DATA_FILE = Path('member_attrition_features.csv')
if not DATA_FILE.exists():
    raise FileNotFoundError('Place an anonymized member_attrition_features.csv file in this directory.')
df = pd.read_csv(DATA_FILE)
print(f'Rows: {len(df):,}; columns: {df.shape[1]:,}')


In [ ]:
df1 = df
df1.columns = map(str.lower, df1.columns)
df1 = df1.fillna(value=np.nan)

In [ ]:
df1.dtypes

In [ ]:
df1.head()

In [ ]:
df.columns

In [ ]:
df1.describe()

In [ ]:
df1.isna().sum()

In [ ]:
df1.columns

In [ ]:
today = pd.to_datetime('today').strftime('%Y-%m-%d')

col_names = df1.columns

for col in col_names:
    if col in df1.filter(regex='count|1m|2m|m3|max|denied', axis=1):
        df1.fillna(0 , inplace = True)
for col in col_names:
    if col in df1.filter(regex='firstope|lastclo', axis=1):
        df1[col].replace('2999-12-31', today)      
    if col in df1.filter(regex='firstope|lastclo', axis=1):
        df1[col] = pd.to_datetime(df1[col], errors = 'coerce').fillna(today)
    
        
df1.head(100)


In [ ]:
df1.dtypes

In [ ]:
df1['auto_tenure'] = (df1['autolastclosed'] - df1['autofirstopened']).dt.days.astype(np.int64) // 30.4
df1['personal_tenure'] = (df1['personallastclosed'] - df1['personalfirstopened']).dt.days.astype(np.int64) // 30.4
df1['other_tenure'] = (df1['otherlastclosed'] - df1['otherfirstopened']).dt.days.astype(np.int64) // 30.4
df1['cc_tenure'] = (df1['creditcardlastclosed'] - df1['credictcardfirstopened']).dt.days.astype(np.int64) // 30.4

df1['sav_tenure'] = (df1['savingslastclosed'] - df1['savingsfirstopened']).dt.days.astype(np.int64) // 30.4
df1['check_tenure'] = (df1['checkinglastclosed'] - df1['checkingfirstopened']).dt.days.astype(np.int64) // 30.4
df1['cert_tenure'] = (df1['certificatelastclosed'] - df1['certificatefirstopened']).dt.days.astype(np.int64) // 30.4
df1['ira_tenure'] = (df1['iralastclosed'] - df1['irafirstopened']).dt.days.astype(np.int64) // 30.4

#df1.head(50)
df1.dtypes

In [ ]:
clean_df = df1[df1.columns.drop(list(df.filter(regex='firstope|lastclo|date|yearmonthsid')))]

cols = list(clean_df)
cols.insert(-1, cols.pop(cols.index('churn')))
cols.insert(-2, cols.pop(cols.index('ira_tenure')))
clean_df = clean_df.loc[:, cols]

clean_df.head()

In [ ]:
target = clean_df[['churn']]
final = clean_df.drop(['identifier', 'membernumber'], axis = 1)

In [ ]:
final.dtypes

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report

In [ ]:
xtrain, xtest, ytrain, ytest = train_test_split(final, target, test_size=0.2, random_state = 5, stratify=target)

print('Training Features Shape:',xtrain.shape)
print('Training Labels Shape:',ytrain.shape)
print('Testing Features Shape:',xtest.shape)
print('Testing Labels Shape:',ytest.shape)

In [ ]:
lr1 = LogisticRegression()
lr1.fit(xtrain, ytrain)

In [ ]:
pred1 = lr1.predict(xtest)
matrix = confusion_matrix(ytest, pred1)
print(matrix)

In [ ]:
print(classification_report(ytest, lr1.predict(xtest)))

In [ ]:
score = lr1.score(xtest, ytest)
print(score)

In [ ]:
from sklearn import preprocessing
from sklearn.model_selection import GridSearchCV

In [ ]:
#param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000] }
#clf = GridSearchCV(LogisticRegression(penalty='l2'), param_grid)
#GridSearchCV(cv=10,
             #estimator=LogisticRegression(C=1.0, intercept_scaling=1,   
              # dual=False, fit_intercept=True, penalty='l2', tol=0.0001),
             #param_grid={'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000]})

In [ ]:
param_grid1 = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    'penalty': ['l1', 'l2'],
    'max_iter': list(range(100,800,100)),
}
lr_search = GridSearchCV(lr1, param_grid=param_grid1, cv = 10, refit = True)

# fitting the model for grid search 
lr_search.fit(xtrain , ytrain)
lr_search.best_params_
# summarize
print('Mean Accuracy: %.3f' % lr_search.best_score_)
print('Config: %s' % lr_search.best_params

In [ ]:
param_grid1 = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'max_iter': list(range(100,800,100)),
}
lr_search = GridSearchCV(lr1, param_grid=param_grid1, cv = 10, refit = True)

# fitting the model for grid search 
lr_search.fit(xtrain , ytrain)
lr_search.best_params_
# summarize
print('Mean Accuracy: %.3f' % lr_search.best_score_)
print('Config: %s' % lr_search.best_params